# 7. Running a whole search, and reading what comes back

The earlier notebooks drive the pieces: the scan kernel, the physics, the score shapes.
This one drives **the whole pipeline** — screen, scan, score, clean, label, pack, write
— as an ordinary Python call, and then reads the result properly.

It is also the place the **full Arequipa DEM** run belongs. That run is at the end,
guarded so this notebook still executes without the DEM, which is gitignored and a
quarter of a gigabyte.

Three things are worth knowing before the first call:

- **Everything the command line can do, the library can do.** Configuration files,
  the memory pre-flight, the run summary. There is no CLI-only behaviour left.
- **The pipeline returns its results.** It used to return `None` and leave callers to
  find and re-read the JSON it had just written.
- **It explains itself.** A plain-language account of what was found and why, printed
  and saved as `explanation.txt`, on by default.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import numpy as np

import contextlib
import io
import json
import tempfile

import explain
import site_searcher as ss

WORK = tempfile.mkdtemp(prefix="oroscope_nb07_")
print("working in", WORK)

working in /tmp/oroscope_nb07_bkt6dqif


## Configuration is data, not a command-line concern

`default_config()` returns every knob the tool understands, with its default.
`generate_config(path, preset)` writes that as a template, and `load_config(path)` reads
one back. All three used to exist only inside `main()`, reachable by running the CLI.

A template naming **every** key matters more than it sounds: a config with holes in it
falls back silently for whatever it omits, and the fallback file is the least visible
input the tool has.

In [2]:
cfg = ss.default_config("arequipa")
print(f"{len(cfg)} keys, e.g.:")
for key in ("dem_path", "min_slope_deg", "max_slope_deg", "candidate_stride",
            "downsample_factor", "min_score", "explain"):
    print(f"   {key:>20}: {cfg[key]!r}")

path = os.path.join(WORK, "arequipa.json")
ss.generate_config(path, "arequipa")
print(f"\nwritten and read back identically: {ss.load_config(path) == cfg}")

79 keys, e.g.:
               dem_path: 'arequipa_SRTMGL1.tif'
          min_slope_deg: 3.0
          max_slope_deg: 25.0
       candidate_stride: 5
      downsample_factor: 4
              min_score: 0.0
                explain: True

written and read back identically: True


## Before a big run: what will it cost?

`estimate_peak_memory_gb` predicts the *anonymous* allocations from the DEM's size and
two parameters. The memory-mapped DEM is deliberately excluded — it is file-backed and
the kernel can evict it, and counting it would make every large search look impossible
when the streaming design exists precisely so that it is not.

`downsample_factor` is the knob that matters: the labelling arrays scale as its inverse
square. Here is the real Arequipa DEM, 10204 × 12603 pixels.

In [3]:
rows, cols = 10204, 12603
print(f"Arequipa DEM: {rows} x {cols} = {rows*cols/1e6:.0f} Mpx\n")
for ds in (1, 2, 4, 8):
    need = ss.estimate_peak_memory_gb(rows, cols, downsample_factor=ds)
    print(f"   downsample_factor {ds}:  {need:5.2f} GiB")

have = ss.available_memory_gb()
print(f"\navailable right now: {have:.1f} GiB" if have else "\n(memory not reportable here)")
print("\nThis is why the full run uses downsample_factor 4.")

Arequipa DEM: 10204 x 12603 = 129 Mpx

   downsample_factor 1:   4.45 GiB
   downsample_factor 2:   2.74 GiB
   downsample_factor 4:   2.32 GiB
   downsample_factor 8:   2.21 GiB

available right now: 7.2 GiB

This is why the full run uses downsample_factor 4.


The estimate is rough and says so — `survival_fraction` is the share of pixels passing
the topographic screen, which is terrain-dependent and unknown until the screen has run.
It is meant to catch the order-of-magnitude mistake, not to predict a number.

`preflight_memory` does the whole job: estimate, warn if it is close, and cap the
process's address space so a search that outgrows the machine fails with `MemoryError`
naming itself rather than letting the kernel's OOM killer pick a victim — which may be
your editor. A ten-point sweep once did exactly that at 6.9 GB.

## A complete run

Synthetic terrain, so this executes anywhere. A ridge with a slope in front of it: the
slope sees the ridge, and the ridge's own flank sees the terrain rising beyond.

In [4]:
def ridge_and_slope(n, cell_x):
    """A valley between a ridge and a rising slope. Closed-form, no DEM needed."""
    cols = np.arange(n, dtype=np.float64)[None, :].repeat(n, 0)
    x = cols * cell_x
    ridge = 1400.0 * np.exp(-((x - 0.30 * n * cell_x) / (0.05 * n * cell_x)) ** 2)
    rise = np.clip((x - 0.55 * n * cell_x) / (0.45 * n * cell_x), 0, 1) ** 2 * 1500.0
    return (2200.0 + ridge + rise).astype(np.float32)

In [5]:
import tifffile as tiff

grid = ss.resolve_grid_geometry("no-such-file.tif", -15.6, cell_size_deg=1/3600)
z = ridge_and_slope(700, grid.cell_size_x)

dem = os.path.join(WORK, "ridge.tif")
tiff.imwrite(dem, z, extratags=[
    (33550, "d", 3, (1/3600, 1/3600, 0.0)),                    # ModelPixelScale
    (33922, "d", 6, (0.0, 0.0, 0.0, -72.3, -15.6, 0.0)),       # ModelTiepoint
])
print("wrote a GeoTIFF carrying its own resolution and corner:", os.path.basename(dem))

wrote a GeoTIFF carrying its own resolution and corner: ridge.tif


Now the search itself. Note what is *not* passed: no origin — the DEM carries its own
corner in the tiepoint tag, and reading it removes the most error-prone input the tool
has. A supplied origin that disagrees with the file by more than ~100 m is reported
rather than silently honoured, because a wrong origin does not fail, it
mis-georeferences every output.

The run prints a great deal. It is captured here and unpacked below.

In [6]:
log = io.StringIO()
with contextlib.redirect_stdout(log), contextlib.redirect_stderr(io.StringIO()):
    results = ss.find_grand_regions_interactive(
        dem_path=dem,
        run_output_dir=os.path.join(WORK, "run"),
        # Note search_mode and grid_type. The function's own defaults are 'single'
        # and 'square'; the config template's are 'distributed' and 'hex'. Omitting
        # them here is not the same as omitting them from a config file -- see below.
        search_mode="distributed", grid_type="hex",
        target_antennas=200, min_sub_array_size=20,
        min_width_km=1.0, antenna_spacing_km=1.0,
        min_dist_km=3.0, max_dist_km=20.0,
        downsample_factor=2, tile_size=256, candidate_stride=5, num_cores=2,
    )

print(f"{len(log.getvalue().splitlines())} lines of output captured\n")
print("returned:", ", ".join(sorted(results)))

267 lines of output captured

returned: aperture, explanation, funnel, mode, output_files, parameters, provenance, regions, results, timestamp, timings_sec


### One trap worth knowing

**The function's defaults are not the config template's defaults.** Five parameters
differ, and omitting one means different things depending on which door you came in by:

| parameter | `find_grand_regions_interactive` | `default_config()` |
|---|---|---|
| `search_mode` | `single` | `distributed` |
| `grid_type` | `square` | `hex` |
| `target_antennas` | 1000 | 10000 |
| `min_dist_km` | 30.0 | 10.0 |
| `min_sub_array_size` | 100 | 500 |

An earlier draft of this notebook omitted `search_mode` and quietly ran a *single*
search with a 30 km minimum distance, which on this small ridge found nothing at all.
The funnel said so plainly, which is the system working — but the safe habit when
driving the library is to start from `ss.default_config()` and override, rather than to
rely on the signature's defaults.

In [7]:
template = ss.default_config()
signature_defaults = {
    "search_mode": "single", "grid_type": "square",
    "target_antennas": 1000, "min_dist_km": 30.0, "min_sub_array_size": 100,
}
print(f"{'parameter':22} {'function':>12} {'config template':>18}")
for key, value in signature_defaults.items():
    print(f"{key:22} {str(value):>12} {str(template[key]):>18}")

parameter                  function    config template
search_mode                  single        distributed
grid_type                    square                hex
target_antennas                1000              10000
min_dist_km                    30.0               10.0
min_sub_array_size              100                500


That dictionary is the same content the results JSON holds, plus the explanation and
the paths written. No re-reading the file it just wrote.

In [8]:
print(f"sites:    {results['results']['total_sites']}")
print(f"capacity: {results['results']['total_capacity']}")
print(f"stages:   {', '.join(results['timings_sec'])}")
print(f"files:    {len(results['output_files'])} written")
for f in results["output_files"]:
    print("   ", os.path.basename(f))

sites:    1
capacity: 180
stages:   load_dem, topographic_screen, ray_tracing, morphology, capacity_analysis, outputs, total
files:    5 written
    oroscope_results_ridge.tif
    oroscope_results_ridge.tfw
    oroscope_results_ridge.png
    oroscope_results_ridge.json
    provenance.json


## The funnel is the diagnostic

Every filter records how many pixels survived it. When a search returns little or
nothing, **the stage where the count collapses is the constraint responsible** — and
that is the single most useful thing anyone can be told about a disappointing run.

In [9]:
for stage, count in results["funnel"].items():
    print(f"   {stage:<34} {count:>12,}")

binding = explain.binding_constraint(results["funnel"])
print(f"\nbinding constraint: {binding['stage']!r}")
print(f"   kept {100*binding['kept_fraction']:.1f}% of the {binding['before']:,} that reached it")
print(f"   change: {binding['knob']}")

   DEM pixels                              490,000
   finite elevation                        490,000
   slope 3.0-25.0 deg                      226,100
   kept by stride 5                         45,224
   directions accepted                      43,684
   after gap closing                       198,024
   after pruning (< 1.0 km wide)           194,010
   pixels in selected sites (est.)         164,820

binding constraint: 'slope 3.0-25.0 deg'
   kept 46.1% of the 490,000 that reached it
   change: min_slope_deg / max_slope_deg


Two stages are excluded from that search by construction, and it is worth knowing why:

- **`kept by stride N`** is a deliberate subsample, not a filter. It removes four
  candidates in five and the acceptance is unchanged, so calling it the constraint
  would name the same answer on nearly every run.
- **`after gap closing`** *adds* pixels. A stage that grows the set cannot be what
  shrank it.

## Which sites are actually in the result

`sites` lists everything that cleared the area and capacity thresholds. With
`stop_at_target`, selection walks that capacity-sorted list until the target is met and
stops — so the list can be longer than the result. Only the selection is in
`total_sites`, `total_capacity` and the exported raster.

Each record says which it is.

In [10]:
log2 = io.StringIO()
with contextlib.redirect_stdout(log2), contextlib.redirect_stderr(io.StringIO()):
    truncated = ss.find_grand_regions_interactive(
        dem_path=dem, run_output_dir=os.path.join(WORK, "run2"),
        target_antennas=50, min_sub_array_size=5, stop_at_target=True,
        min_width_km=1.0, antenna_spacing_km=1.0,
        min_dist_km=3.0, max_dist_km=20.0,
        downsample_factor=2, tile_size=256, candidate_stride=5, num_cores=2,
    )

chosen, shortlisted = explain.selected_sites(truncated)
print(f"listed in the file: {len(chosen) + len(shortlisted)}")
print(f"selected:           {truncated['results']['total_sites']}\n")
for site in chosen + shortlisted:
    mark = "selected" if site["selected"] else "not selected"
    print(f"   site {site['site_id']:>3}  {site['area_km2']:>8.2f} km²  "
          f"{site['capacity_exact']:>5} detectors   {mark}")

print(f"\nsumming everything listed:  {sum(s['area_km2'] for s in chosen + shortlisted):8.2f} km²")
print(f"summing the selection:      {sum(s['area_km2'] for s in chosen):8.2f} km²  <- the raster")

listed in the file: 1
selected:           1

   site   3    150.74 km²    168 detectors   selected

summing everything listed:    150.74 km²
summing the selection:        150.74 km²  <- the raster


Totalling the wrong one over-reports, which is exactly the mistake this flag exists to
prevent. The sites that were not selected are the *next best ground*, not ground that
failed — worth keeping in the file, worth excluding from the totals.

## Attribution: what held each site back

The score is a product of **named** components, each in [0, 1], and each site's record
carries the distribution of every one. Under a product the lowest component bounds the
total from above, so naming it turns "this site scored 0.34" into something actionable.

In [11]:
site = chosen[0]
scan = site["arrival_scan"]
parts = {k[len("score_"):-len("_p50")]: v for k, v in scan.items()
         if k.startswith("score_") and k.endswith("_p50") and k != "score_p50"}

print(f"site {site['site_id']}, median score {scan['score_p50']:.3f}\n")
for name, value in sorted(parts.items(), key=lambda kv: kv[1]):
    bar = "#" * int(round(value * 40))
    print(f"   {name:>14}  {value:5.3f}  {bar}")

name, value = explain.weakest_component(scan)
print(f"\nweakest: {name} at {value:.3f}")

site 3, median score 0.341

        footprint  0.447  ##################
      geomagnetic  0.865  ###################################
      solid_angle  0.888  ####################################
            depth  1.000  ########################################
         distance  1.000  ########################################
           shower  1.000  ########################################

weakest: footprint at 0.447


On the real Colca configurations this is unambiguous: `solid_angle` is the weakest
component at **15 of 15** TAMBO sites, with everything else at 1.0 except the decay term
at 0.96. So that result is set almost entirely by `solid_angle_half_sr`, whose 0.05 sr
default is a GRAND-scale value.

That is the kind of statement the components make available and a single total does not.

## How much did closing move the area?

The reported area is not the physics-accepted area: the mask is closed morphologically
before areas are measured. The published figure is 2.29× at Colca, measured against a
stride-1 control — but each run has the number in it, as closed pixels over
stride-corrected accepted pixels.

In [12]:
ratio = explain.closing_inflation(results["funnel"],
                                 results["parameters"]["candidate_stride"])
print(f"this run: closing moved the mask by {ratio:.2f}x")
print("\nOn the real configurations:")
print("   GRAND Colca  2.19x   (against 2.29x from a stride-1 control -- an independent check)")
print("   TAMBO Colca  0.53x   (a 100 m element cannot bridge the gaps stride 5 leaves,")
print("                        so its area is a LOWER bound, not an upper one)")

this run: closing moved the mask by 0.91x

On the real configurations:
   GRAND Colca  2.19x   (against 2.29x from a stride-1 control -- an independent check)
   TAMBO Colca  0.53x   (a 100 m element cannot bridge the gaps stride 5 leaves,
                        so its area is a LOWER bound, not an upper one)


## The run, explained — the whole summary, here

Everything above is assembled for you. `explain.explain_results` takes the results
dictionary and returns a string — it opens no files, runs nothing and needs no DEM, so
a run from months ago can still be explained from its JSON.

It is on by default, printed at the end of every run and saved as `explanation.txt`
beside the results, because these runs are meant to be handed to other people and a
terminal scrollback is not. `--no_explain` suppresses it.

This is the text in full. Its sections, in order:

| section | answers |
|---|---|
| **The run** | what was searched, at what resolution, by which commit |
| **The headline** | how many sites, how much area, how many detectors |
| **Where the candidates went** | the funnel, and **which constraint bound this run** |
| **From pixels to sites** | labelled regions → area threshold → capacity threshold |
| **The sites** | each one's area, capacity, facing, score and weakest criterion |
| **Why these sites qualify** | what the ground actually offers, criterion by criterion, with coordinates |
| **What energy this geometry favours** | where the geometric aperture peaks |
| **How to read these numbers** | the closing factor *for this run*, and what area is not |
| **Which of these are assumptions** | choices rather than measurements, with measured sensitivities |
| **What to try next** | concrete commands, chosen from what this run did |

In [13]:
print(results["explanation"])

 WHAT THIS SEARCH FOUND, AND WHY

THE RUN
-------
  DEM              /tmp/oroscope_nb07_bkt6dqif/ridge.tif
  Origin           -15.600000, -72.300000  (auto-detected from the GeoTIFF tiepoint)
  Resolution       0.00027778°/px  =  30.7 m N-S x 29.8 m E-W
  Layout           distributed search, hex grid, 1.0 km spacing
  Finished         2026-08-16 00:31:55
  Code             commit ff7bd4b on dev (dirty tree)
  DEM checksum     sha256 d53de4fdc2b72326…
  Command          /home/mbustamante/anaconda3/envs/sssearch/lib/python3.12/site-packages/ipykernel_launcher.py -f /tmp/tmp5n_sq4kq.json --HistoryManager.hist_file=:memory:

THE HEADLINE
------------
  1 site covering 150.7 km², 180 detectors against a target of 200.

  Largest by capacity: site 3, 150.74 km², 180 detectors, facing W, centred
  -15.6972, -72.1444 — paste that into a map.

  The target of 200 was not reached. The funnel below says why; the shortfall
  is 20 detectors, 10% of the target.

WHERE THE CANDIDATES WENT
----------

## Provenance

Separate from the science outputs, and the answer to "what produced this number?".

In [14]:
prov = results["provenance"]
print(f"commit:   {prov['git']['commit'][:10]} on {prov['git']['branch']}"
      f"  ({'dirty' if prov['git']['dirty'] else 'clean'} tree)")
print(f"DEM:      {os.path.basename(prov['dem']['path'])}")
print(f"          sha256 {prov['dem']['sha256'][:24]}...")
print(f"          {prov['dem']['cell_size_y_m']:.2f} m N-S x {prov['dem']['cell_size_x_m']:.2f} m E-W")
print(f"python:   {prov['platform']['python']} on {prov['platform']['system']}")
print(f"packages: {', '.join(f'{k} {v}' for k, v in list(prov['packages'].items())[:4])}, ...")

commit:   ff7bd4bb5d on dev  (dirty tree)
DEM:      ridge.tif
          sha256 d53de4fdc2b72326673978ad...
          30.72 m N-S x 29.77 m E-W
python:   3.12.12 on Linux 7.0.0-28-generic
packages: numpy 2.3.4, scipy 1.16.3, numba 0.62.1, tifffile 2021.7.2, ...


---

## The full Arequipa DEM

Every number this project has published comes from **crops** — Colca, and small Arequipa
windows. The full DEM is the run that has never been done, and this is where it lands.

**Read, not run.** The cells below open results that were produced locally and stored in
`results/arequipa_full/`. They do not start a search. Each of these searches takes about
half an hour, CI executes notebooks on every push, and a tutorial costing ninety minutes
of compute per commit is a bill rather than a tutorial. The expensive half runs once, on
a machine that has the DEM; the notebook opens a few hundred kilobytes of JSON.

To produce or refresh the store:

```bash
python tools/run_arequipa_full.py --dry-run   # what it will do, and what it will cost
python tools/run_arequipa_full.py             # GRAND, TAMBO, then the combination
```

**Regenerate it when a configuration changes, and not otherwise.** The store carries a
manifest naming the configs and the time, so a stale one is detectable rather than
merely suspected.

Three searches, all at the same `downsample_factor` so their masks are pixel-aligned:

| | config | what it asks |
|---|---|---|
| **GRAND alone** | `config/grand_arequipa_full.json` | 3–25° deployable ground seeing a target 10–40 km away, within ±3° of the horizon |
| **TAMBO alone** | `config/tambo_arequipa_full.json` | a 20–60° near wall facing a ≥25° far wall, 2–5 km across |
| **Combined** | `combine_experiments` over both | joint, union, and how much of each sits inside the other |

**What it costs.** 10204 × 12603 pixels, about 129 Mpx. At `downsample_factor: 4` the
estimator says 2.3 GiB against the ~6 GiB typically free; at 1 it says 4.5 GiB, which is
why 4 is the setting. That choice has a price worth stating: area is measured on the
downsampled mask while capacity is measured at full resolution, so a feature a few
pixels wide keeps its detectors and loses area. **Read these areas as lower bounds**,
and more so for TAMBO's canyon strips than for GRAND's blobs.

In [15]:
STORE = os.path.abspath(os.path.join("..", "results", "arequipa_full"))

def load_stored(label):
    """Reads one stored run, or returns None when the store does not have it yet."""
    path = os.path.join(STORE, f"{label}_results.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

manifest_path = os.path.join(STORE, "manifest.json")
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"store generated {manifest['generated']} by {manifest['generated_by']}")
    print(f"from {manifest['dem']}\n")
    for name in manifest["files"]:
        print("   ", name)
else:
    manifest = None
    print("The full-DEM store is empty: these searches have not been run yet.\n")
    print("Produce it with:")
    print("    python tools/run_arequipa_full.py --dry-run")
    print("    python tools/run_arequipa_full.py")

grand_full = load_stored("grand")
tambo_full = load_stored("tambo")

The full-DEM store is empty: these searches have not been run yet.

Produce it with:
    python tools/run_arequipa_full.py --dry-run
    python tools/run_arequipa_full.py


### GRAND over the whole DEM

The four things worth reading, in this order:

1. **The funnel**, and specifically whether the binding constraint is the same one the
   crops found. If a full DEM is bound by a different stage than its crops were, the
   crops were not representative and every number derived from them needs re-reading.
2. **The area**, against the crop scaled up — and against the closing factor this run
   reports for itself, rather than the 2.29× quoted from Colca.
3. **The site count and their spread.** A crop cannot say whether the good ground is one
   region or fifty scattered ones, and that is a deployment question, not a physics one.
4. **The weakest score component**, which on the crops is `solid_angle` everywhere. If
   that holds at full scale it is a statement about the criterion, not about Peru.

In [16]:
def summarise(results, label):
    if results is None:
        print(f"{label}: not in the store yet.")
        return
    chosen, shortlisted = explain.selected_sites(results)
    area = sum(s["area_km2"] for s in chosen)
    binding = explain.binding_constraint(results["funnel"])
    ratio = explain.closing_inflation(results["funnel"],
                                      results["parameters"]["candidate_stride"])
    weakest = [explain.weakest_component(s.get("arrival_scan") or {}) for s in chosen]
    named = [w[0] for w in weakest if w]

    print(f"{label}")
    print(f"   sites          {results['results']['total_sites']:>10,}"
          f"   ({len(shortlisted)} more cleared the thresholds, not selected)")
    print(f"   capacity       {results['results']['total_capacity']:>10,}")
    print(f"   area           {area:>10,.1f} km²")
    if binding:
        print(f"   bound by       {binding['stage']}"
              f"  (kept {100*binding['kept_fraction']:.1f}%)")
    if ratio is not None:
        print(f"   closing moved  {ratio:>10.2f}x")
    if named:
        commonest = max(set(named), key=named.count)
        print(f"   weakest        {commonest} at {named.count(commonest)}/{len(named)} sites")

summarise(grand_full, "GRAND, full Arequipa DEM")

GRAND, full Arequipa DEM: not in the store yet.


In [17]:
if grand_full is not None:
    print(grand_full.get("explanation") or explain.explain_results(grand_full))
else:
    print("Nothing stored for GRAND yet -- see the cell above for how to produce it.")

Nothing stored for GRAND yet -- see the cell above for how to produce it.


### TAMBO over the whole DEM

The more interesting of the two, because TAMBO's criteria are canyon-shaped and the
crop was *chosen* for containing a canyon. Over the whole DEM the question becomes: how
much other canyon is there, and is any of it as good?

Note that the areas here are the ones most affected by `downsample_factor: 4`: a strip
along a wall is exactly the feature that loses area to downsampling while keeping its
detectors.

In [18]:
summarise(tambo_full, "TAMBO, full Arequipa DEM")

TAMBO, full Arequipa DEM: not in the store yet.


In [19]:
if tambo_full is not None:
    print(tambo_full.get("explanation") or explain.explain_results(tambo_full))
else:
    print("Nothing stored for TAMBO yet -- see above for how to produce it.")

Nothing stored for TAMBO yet -- see above for how to produce it.


### Where both are viable

The overlay. On the Colca crop the answer was decided by slope: GRAND's 3–25° deployable
band against Colca's ~40° walls leaves only a 20–25° sliver, so the joint was about a
percent of GRAND's area and three fifths of TAMBO's.

Whether that survives at full scale is a real question. The crop contains one canyon
system; the DEM contains many, of varying wall slope, and the joint area is the
programme-level number — one site, one road, one power feed, two experiments.

In [20]:
report_path = os.path.join(STORE, "combined_report.json")
if not os.path.exists(report_path):
    print("The combination has not been produced yet.")
else:
    with open(report_path) as f:
        report = json.load(f)

    width = max(len(r["label"]) for r in report["runs"])
    print(f"   {'experiment'.ljust(width)} {'area km²':>12} {'sites':>7} "
          f"{'capacity':>10} {'in joint':>9}")
    print("   " + "-" * (width + 42))
    for r in report["runs"]:
        print(f"   {r['label'].ljust(width)} {r['area_km2']:>12,.1f} "
              f"{r['reported_sites']:>7,} {r['reported_capacity']:>10,} "
              f"{100*r['fraction_of_own_area_in_joint']:>8.1f}%")
    print("   " + "-" * (width + 42))
    print(f"   joint  {report['joint']['area_km2']:>10,.1f} km²")
    print(f"   union  {report['union']['area_km2']:>10,.1f} km²")
    for pair, stats in report["pairwise_overlap"].items():
        print(f"   {pair}: Jaccard {stats['jaccard']:.4f}")

The combination has not been produced yet.


And the overlay explains itself too, the same way a search does — including the part
that is easy to get wrong. Co-location is decided by whichever *ground* property the
two experiments share least of, because a pixel has one slope and both have to accept
it. What each asks of the **view** — the distance window, the arrival elevations — may
differ freely: two experiments can look out from the same hillside at different ranges
without conflict.

`oroscope-combine` prints this and saves it as `combination_explanation.txt`.

In [21]:
if not os.path.exists(report_path):
    print("The combination has not been produced yet.")
else:
    runs = {label: res for label, res in (("GRAND", grand_full), ("TAMBO", tambo_full))
            if res is not None}
    print(explain.explain_combination(report, runs))

The combination has not been produced yet.


### Comparing against the crop

The crop's numbers, for reference — GRAND 4580.2 km² in 1 site with 5317 detectors,
TAMBO 83.6 km² in 15 sites with 9717, joint 50.1 km². If the full DEM's binding
constraint or weakest component differs from these, the crop was not representative and
the comparison is the finding.

A crop is chosen because it is interesting. **A search over ground chosen for being
interesting is not a survey**, and that is the gap this run exists to close.

In [22]:
if grand_full is not None and tambo_full is not None:
    print("Both full-DEM runs are stored; compare their funnels against the crops':\n")
    for label, res in (("GRAND", grand_full), ("TAMBO", tambo_full)):
        b = explain.binding_constraint(res["funnel"])
        print(f"   {label:>6}: bound by {b['stage']!r} "
              f"(kept {100*b['kept_fraction']:.1f}%)")
    print("\n   Colca crop, for comparison:")
    print("   GRAND: bound by 'directions accepted' (kept 60.1%)")
    print("   TAMBO: bound by 'directions accepted' (kept 17.5%)")
else:
    print("Run both searches to make this comparison.")

Run both searches to make this comparison.


---

*Part of the [Oroscope](https://github.com/mbustama/oroscope) tutorials. Previous: [Combining and sensitivity](06_combining_and_sensitivity.ipynb). Full API reference: [oroscope docs](https://mbustama.github.io/oroscope/functions.html).*